# Classical ML for Binary Diabetes Risk Prediction

**Course:** MDS 5 – Predictive Analytics  
**Notebook:** `diabetes_ml_benchmark.ipynb` (final acceptance notebook)  
**Dataset:** `diabetes_binary_50k_stratified.csv` (50,000 stratified rows)  
**Target:** `Diabetes_binary` (0 = no diabetes, 1 = prediabetes or diabetes)

Style follows the course demo: **small numbered cells**, one step at a time.  
Issue progress is built section by section in this same notebook.

> Start the kernel with working directory = project root (`Income_prediction/`).

## Data Ingestion and Quality Checks

Load the stratified binary BRFSS sample and run basic quality checks before EDA.

In [1]:
# 1. Import core libraries
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from IPython.display import display
except ImportError:
    display = print

In [2]:
# 2. Resolve project root and create export folders
# 论文用图/表写入 files/pic 与 files/data，供后续描述性分析引用
cwd = Path.cwd().resolve()
ROOT = None
for p in [cwd, *cwd.parents]:
    if (p / "dataset" / "diabetes_binary_50k_stratified.csv").exists():
        ROOT = p
        break
if ROOT is None:
    raise FileNotFoundError(
        "Cannot find dataset/diabetes_binary_50k_stratified.csv. "
        "Start kernel from project root, or run dataset/make_binary_50k.py first."
    )

DATA_PATH = ROOT / "dataset" / "diabetes_binary_50k_stratified.csv"
PIC_DIR = ROOT / "files" / "pic"
DATA_DIR = ROOT / "files" / "data"
PIC_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "Diabetes_binary"
RANDOM_STATE = 42

print("ROOT:", ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())

ROOT: D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction
DATA_PATH exists: True


In [3]:
# 3. Load CSV
df = pd.read_csv(DATA_PATH)
print("Raw shape:", df.shape)
df.head()

Raw shape: (50000, 22)


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1.0,1.0,1.0,21.0,1.0,0.0,0.0,1.0,1.0,0.0,...,0.0,3.0,10.0,0.0,0.0,0.0,13.0,4.0,4.0,0
1,1.0,1.0,1.0,33.0,1.0,0.0,0.0,1.0,0.0,1.0,...,0.0,3.0,30.0,14.0,1.0,0.0,7.0,5.0,1.0,1
2,0.0,0.0,1.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,30.0,30.0,1.0,0.0,4.0,4.0,5.0,1
3,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,3.0,15.0,0.0,1.0,0.0,10.0,6.0,4.0,0
4,0.0,1.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,2.0,0.0,0.0,0.0,0.0,1.0,5.0,7.0,0


In [4]:
# 4. Basic quality checks（缺失 / 重复 / 目标分布）
# 即使缺失为 0，也要写进 Methodology，说明做过可靠性检查
print("Missing values (total):", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))
print("Class counts:")
print(df[TARGET].value_counts().sort_index())
print("Class ratio:")
print(df[TARGET].value_counts(normalize=True).sort_index().round(4))

Missing values (total): 0
Duplicate rows: 1997
Class counts:
Diabetes_binary
0    42121
1     7879
Name: count, dtype: int64
Class ratio:
Diabetes_binary
0    0.8424
1    0.1576
Name: proportion, dtype: float64


In [5]:
# 5. Sanity ranges for BMI and unhealthy-day counts
bmi_bad = ~df["BMI"].between(10, 60)
days_bad = ~(df["MentHlth"].between(0, 30) & df["PhysHlth"].between(0, 30))
print("BMI outside [10, 60]:", int(bmi_bad.sum()))
print("MentHlth/PhysHlth outside [0, 30]:", int(days_bad.sum()))

BMI outside [10, 60]: 128
MentHlth/PhysHlth outside [0, 30]: 0


In [6]:
# 6. Drop duplicates for a clean EDA / modeling baseline
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after drop_duplicates:", df.shape)
print("Class ratio after drop_duplicates:")
print(df[TARGET].value_counts(normalize=True).sort_index().round(4))

Shape after drop_duplicates: (48003, 22)
Class ratio after drop_duplicates:
Diabetes_binary
0    0.8362
1    0.1638
Name: proportion, dtype: float64


## Difference vs Adult Income Demo

| Aspect | Adult demo (course) | This project (BRFSS diabetes) |
|--------|---------------------|-------------------------------|
| Target | Income ≤/> $50K | `Diabetes_binary` risk |
| Feature types | Mixed categorical + numeric → **Label Encoding** needed | Already numeric (binary / ordinal / continuous) |
| Main prep focus | Encoding + Pearson Top-k | **Scaling** + Pearson Top-k + **class imbalance** |
| Class balance | Often skewed income classes | Strong imbalance (~84% : ~16%) |

No Label Encoding step is required here; later issues will focus on `StandardScaler` and imbalance-aware metrics.

## Exploratory Data Analysis

Describe class imbalance, key feature distributions, and correlations.  
Figures under `files/pic/` are primary paper Analysis material.

In [7]:
# 1. Import plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["savefig.dpi"] = 150

In [8]:
# 2. Class balance bar chart -> files/pic/class_balance.png
# 描述性分析提示：约 84% vs 16%，Accuracy 会虚高，论文应主看 F1/AUC/Recall
class_counts = df[TARGET].value_counts().sort_index()
class_ratio = df[TARGET].value_counts(normalize=True).sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
labels = ["No diabetes (0)", "Prediabetes/Diabetes (1)"]
bars = ax.bar(labels, class_counts.values, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("Count")
ax.set_title("Class Balance (Diabetes_binary)")
for b, v, r in zip(bars, class_counts.values, class_ratio.values):
    ax.text(
        b.get_x() + b.get_width() / 2,
        b.get_height(),
        f"{v}\n({r:.1%})",
        ha="center",
        va="bottom",
        fontsize=9,
    )
fig.tight_layout()
fig.savefig(PIC_DIR / "class_balance.png", bbox_inches="tight")
plt.show()
print("Saved:", PIC_DIR / "class_balance.png")

Saved: D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction\files\pic\class_balance.png


C:\Users\yingh\AppData\Local\Temp\ipykernel_50788\2652067963.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
# 3. Key descriptive statistics for numeric / ordinal features
# 用 describe 支撑 Methodology 中的数据描述
desc_cols = ["BMI", "MentHlth", "PhysHlth", "Age", "GenHlth", "Education", "Income"]
desc = df[desc_cols].describe().T.round(3)
display(desc)

,count,mean,std,min,25%,50%,75%,max
BMI,48003.0,28.471,6.524,12.0,24.0,27.0,31.0,98.0
MentHlth,48003.0,3.317,7.549,0.0,0.0,0.0,2.0,30.0
PhysHlth,48003.0,4.427,8.846,0.0,0.0,0.0,3.0,30.0
Age,48003.0,8.070,3.076,1.0,6.0,8.0,10.0,13.0
GenHlth,48003.0,2.553,1.061,1.0,2.0,2.0,3.0,5.0
Education,48003.0,5.017,0.992,1.0,4.0,5.0,6.0,6.0
Income,48003.0,5.995,2.077,1.0,5.0,7.0,8.0,8.0


In [10]:
# 4. Selected feature histograms (scale motivation)
# BMI / 天数类特征量纲不同，后续对 LR/KNN/SVM 需要标准化
cont_cols = ["BMI", "MentHlth", "PhysHlth", "Age", "GenHlth", "Income"]
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, col in zip(axes.ravel(), cont_cols):
    sns.histplot(df[col], bins=30, ax=ax, color="#4C72B0")
    ax.set_title(col)
fig.suptitle("Selected Feature Distributions", y=1.02)
fig.tight_layout()
fig.savefig(PIC_DIR / "feature_histograms.png", bbox_inches="tight")
plt.show()
print("Saved:", PIC_DIR / "feature_histograms.png")

Saved: D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction\files\pic\feature_histograms.png


C:\Users\yingh\AppData\Local\Temp\ipykernel_50788\793415727.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [11]:
# 5. Boxplots of key features by diabetes class
# 观察阳性类在 BMI、GenHlth 等变量上是否整体偏高
box_cols = ["BMI", "GenHlth", "Age", "Income"]
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.ravel(), box_cols):
    sns.boxplot(data=df, x=TARGET, y=col, ax=ax, color="#55A868")
    ax.set_title(f"{col} by {TARGET}")
    ax.set_xlabel(TARGET)
fig.suptitle("Key Features by Diabetes Class", y=1.02)
fig.tight_layout()
fig.savefig(PIC_DIR / "feature_boxplots_by_class.png", bbox_inches="tight")
plt.show()
print("Saved:", PIC_DIR / "feature_boxplots_by_class.png")

Saved: D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction\files\pic\feature_boxplots_by_class.png


C:\Users\yingh\AppData\Local\Temp\ipykernel_50788\721221721.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Pearson Correlation (EDA)

Mirror the course demo style: compute correlation → inspect |corr| with target → heatmap.  
Train-only Top-k feature selection is deferred to **Issue #3**.

In [12]:
# 1. Compute Pearson correlation matrix (full cleaned frame)
corr_matrix = df.corr(method="pearson")
print("Correlation matrix shape:", corr_matrix.shape)

Correlation matrix shape: (22, 22)


In [13]:
# 2. Preview absolute correlations with the target
target_corr = corr_matrix[TARGET].drop(TARGET)
target_corr_abs = target_corr.abs().sort_values(ascending=False)
print("Top-|corr| features vs Diabetes_binary:")
print(target_corr_abs.head(12).round(4))

Top-|corr| features vs Diabetes_binary:
GenHlth                 0.2908
HighBP                  0.2649
BMI                     0.2189
DiffWalk                0.2068
HighChol                0.2046
Age                     0.1849
HeartDiseaseorAttack    0.1744
Income                  0.1650
PhysHlth                0.1620
Education               0.1249
Stroke                  0.1023
PhysActivity            0.1022
Name: Diabetes_binary, dtype: float64


In [14]:
# 3. Plot heatmap -> files/pic/correlation_heatmap.png
# 描述性分析提示：关注与目标强相关的临床/自评健康变量，以及特征间共线
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    square=True,
    ax=ax,
    cbar_kws={"shrink": 0.7},
)
ax.set_title("Pearson Correlation Heatmap")
fig.tight_layout()
fig.savefig(PIC_DIR / "correlation_heatmap.png", bbox_inches="tight")
plt.show()
print("Saved:", PIC_DIR / "correlation_heatmap.png")

Saved: D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction\files\pic\correlation_heatmap.png


C:\Users\yingh\AppData\Local\Temp\ipykernel_50788\264510886.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## EDA Findings (Issue #2)

Paper-ready bullet points from this run (after drop_duplicates, n=48,003):

1. **Strong class imbalance:** class 0 ≈ 83.6% (40,138) vs class 1 ≈ 16.4% (7,865). Accuracy alone is misleading for model ranking.
2. **Top |Pearson| with target:** GenHlth (0.291), HighBP (0.265), BMI (0.219), DiffWalk (0.207), HighChol (0.205), then Age / HeartDiseaseorAttack / Income.
3. **Scale differences:** BMI, MentHlth, and PhysHlth use different units/ranges → motivate StandardScaler for LR / KNN / SVM later.
4. **Boxplots by class:** positive class tends toward higher BMI and worse (higher) GenHlth scores.
5. **Vs Adult demo:** features are already numeric (no Label Encoding); focus shifts to scaling + imbalance-aware metrics (F1 / AUC / Recall).

Exported EDA figures:
- iles/pic/class_balance.png
- iles/pic/feature_histograms.png
- iles/pic/feature_boxplots_by_class.png
- iles/pic/correlation_heatmap.png


In [15]:
# EDA export checklist (Issue #2)
eda_figs = [
    PIC_DIR / "class_balance.png",
    PIC_DIR / "feature_histograms.png",
    PIC_DIR / "feature_boxplots_by_class.png",
    PIC_DIR / "correlation_heatmap.png",
]
for p in eda_figs:
    print(("OK" if p.exists() else "MISSING"), p)

OK D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction\files\pic\class_balance.png
OK D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction\files\pic\feature_histograms.png
OK D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction\files\pic\feature_boxplots_by_class.png
OK D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction\files\pic\correlation_heatmap.png


## Train/Test Split and Pearson Top-K (Issue #3)

Align with the course demo: **split first**, then select Top-K by |Pearson| **on the training set only** (no leakage).  
QC / de-duplication already done in earlier cells.

In [16]:
# 1. Stratified train/test split (80/20) BEFORE feature selection
# 先划分再选特征，避免用全数据相关造成泄漏
from sklearn.model_selection import train_test_split

TOP_K = 8  # 本项目特征约 21 个；对齐课程 Top-k，取 8

feature_cols_all = [c for c in df.columns if c != TARGET]
X_all = df[feature_cols_all]
y = df[TARGET]

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train:", X_train_all.shape, "Test:", X_test_all.shape)
print("Train class ratio:")
print(y_train.value_counts(normalize=True).sort_index().round(4))
print("random_state:", RANDOM_STATE, "| TOP_K:", TOP_K)

Train: (38402, 21) Test: (9601, 21)
Train class ratio:
Diabetes_binary
0    0.8362
1    0.1638
Name: proportion, dtype: float64
random_state: 42 | TOP_K: 8


In [17]:
# 2. Rank features by |Pearson| with target on the TRAINING set only
train_tmp = X_train_all.copy()
train_tmp[TARGET] = y_train.values
corr_train = train_tmp.corr(method="pearson")[TARGET].drop(TARGET)
abs_corr = corr_train.abs().sort_values(ascending=False)

top_features = abs_corr.head(TOP_K).index.tolist()
print(f"Top {TOP_K} selected features (train-only Pearson):")
print(top_features)
print(abs_corr.head(TOP_K).round(4))

Top 8 selected features (train-only Pearson):
['GenHlth', 'HighBP', 'BMI', 'DiffWalk', 'HighChol', 'Age', 'HeartDiseaseorAttack', 'Income']
GenHlth                 0.2900
HighBP                  0.2648
BMI                     0.2208
DiffWalk                0.2090
HighChol                0.2059
Age                     0.1832
HeartDiseaseorAttack    0.1702
Income                  0.1634
Name: Diabetes_binary, dtype: float64


In [18]:
# 3. Build selected-feature table and export -> files/data/selected_features.csv
selected_table = pd.DataFrame(
    {
        "feature": top_features,
        "pearson_corr": corr_train.loc[top_features].values,
        "abs_pearson_corr": abs_corr.loc[top_features].values,
    }
)
display(selected_table.round(4))
selected_path = DATA_DIR / "selected_features.csv"
selected_table.to_csv(selected_path, index=False)
print("Saved:", selected_path)

,feature,pearson_corr,abs_pearson_corr
0,GenHlth,0.2900,0.2900
1,HighBP,0.2648,0.2648
2,BMI,0.2208,0.2208
3,DiffWalk,0.2090,0.2090
4,HighChol,0.2059,0.2059
5,Age,0.1832,0.1832
6,HeartDiseaseorAttack,0.1702,0.1702
7,Income,-0.1634,0.1634


Saved: D:\code\xu-term2\Master_data_science\MDS 5 Predictive Analytics\Income_prediction\files\data\selected_features.csv


In [19]:
# 4. Filter X to Top-K features only（对齐老师 notebook 的 filter 步骤）
X_train = X_train_all[top_features]
X_test = X_test_all[top_features]
print("X_train:", X_train.shape, "X_test:", X_test.shape)
X_train.head()

X_train: (38402, 8) X_test: (9601, 8)


,GenHlth,HighBP,BMI,DiffWalk,HighChol,Age,HeartDiseaseorAttack,Income
19741,1.0,0.0,23.0,0.0,1.0,6.0,0.0,8.0
3970,2.0,1.0,24.0,0.0,0.0,13.0,0.0,8.0
46416,2.0,0.0,30.0,0.0,0.0,3.0,0.0,4.0
42340,2.0,0.0,23.0,0.0,0.0,4.0,0.0,5.0
14962,3.0,1.0,30.0,0.0,0.0,6.0,0.0,8.0


## Feature Scaling (Issue #3)

`StandardScaler` is fit **on the training set only**, then applied to train and test.  
Tree models do not require scaling, but we scale once for a fair multi-model comparison (LR / KNN / Linear SVM especially).

In [20]:
# 5. StandardScaler: fit on train, transform train/test
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaled train mean ~0:", np.round(X_train_scaled.mean(axis=0), 5))
print("Scaled train std ~1:", np.round(X_train_scaled.std(axis=0), 5))
print("Scaler fitted on train only; test transformed with the same scaler.")

Scaled train mean ~0: [-0. -0.  0.  0.  0.  0.  0. -0.]
Scaled train std ~1: [1. 1. 1. 1. 1. 1. 1. 1.]
Scaler fitted on train only; test transformed with the same scaler.


In [21]:
# 6. Issue #3 pipeline checklist
print("OK random_state =", RANDOM_STATE)
print("OK TOP_K =", TOP_K)
print("OK top_features =", top_features)
print("OK selected_features.csv exists:", (DATA_DIR / "selected_features.csv").exists())
print("OK X_train_scaled shape:", X_train_scaled.shape)
print("OK X_test_scaled shape:", X_test_scaled.shape)

OK random_state = 42
OK TOP_K = 8
OK top_features = ['GenHlth', 'HighBP', 'BMI', 'DiffWalk', 'HighChol', 'Age', 'HeartDiseaseorAttack', 'Income']
OK selected_features.csv exists: True
OK X_train_scaled shape: (38402, 8)
OK X_test_scaled shape: (9601, 8)
